In [ ]:
import kagglehub
import torch

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO

import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np
import os
import glob
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms

# class MyDataset(Dataset):

    # def __init__(self, image_dir, mask_dir, transform=None, target_transform=None):

    #     self.image_paths = glob.glob(os.path.join(image_dir, "*.jpg"))  # Get all image paths
    #     self.mask_paths = glob.glob(os.path.join(mask_dir, "*.png"))  # Get all mask paths

    #     self.image_paths.sort()
    #     self.mask_paths.sort()

    #     self.transform = transform
    #     self.target_transform = target_transform

    # def __len__(self):
    #     return len(self.image_paths)

    # def __getitem__(self, idx):
    #     # 🔹 Load the image and mask
    #     image = Image.open(self.image_paths[idx]).convert("RGB")
    #     mask = Image.open(self.mask_paths[idx]).convert("L")

    #     # 🔹 Apply transformations for image
    #     if self.transform:
    #         image = self.transform(image)

    #     # 🔹 Apply transformations for mask
    #     if self.target_transform:
    #         mask = self.target_transform(mask)

    #     mask = remap_mask(mask)

    #     return image, mask  # Return image-mask pair


class MyDataset(Dataset):
  def __init__(self, image_paths, mask_paths, transform=None, target_transform=None):

    self.image_paths = image_paths
    self.mask_paths = mask_paths
    self.transform = transform
    self.target_transform = target_transform

    self.image_paths.sort()
    self.mask_paths.sort()

  def __len__(self):
    # TODO: Return the number of samples in the dataset
    # YOUR CODE HERE
    return len(self.image_paths)

  def __getitem__(self, idx):
    # TODO: Load the image and mask at index idx
    # Hint: Use Image.open() and convert image to "RGB", mask to "L" (grayscale)

    # YOUR CODE HERE
    image = Image.open(self.image_paths[idx]).convert("RGB")
    mask = Image.open(self.mask_paths[idx]).convert("L")

    # Apply transforms
    if self.transform:
      image = self.transform(image)

    if self.target_transform:
      mask = self.target_transform(mask)
      mask = remap_mask(mask)  # Convert to binary mask

    return image, mask




from torch.utils.data import DataLoader
from torch import nn
from sklearn.model_selection import train_test_split

# Define transforms for images and masks
image_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Standard ImageNet normalization
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),  # Keep segmentation masks intact
    transforms.PILToTensor(),
])

image_paths = []
mask_paths = []
dataset_root = os.path.join(path, "dataset")

# Loop through all patient folders
for fldr in os.listdir(dataset_root):
  pth = os.path.join(dataset_root, fldr)

  if os.path.isdir(pth):
    for filename in os.listdir(pth):
      # TODO: Check if the file is a mask (contains "_mask")
      # If it's a mask, add to mask_paths
      # Otherwise, if it's a .tif file, add to image_paths

      # YOUR CODE HERE
      if ".png" in filename:
        mask_paths.append(os.path.join(pth, filename))
      elif filename.endswith(".jpg"):
        image_paths.append(os.path.join(pth, filename))

print(f"Total images: {len(image_paths)}")
print(f"Total masks: {len(mask_paths)}")




# Split into train and test sets (80% train, 20% test)

# YOUR CODE HERE
train_images, test_images, train_masks, test_masks = train_test_split(
  image_paths, mask_paths, test_size=0.2, random_state=42
)

# Create Dataset objects
train_dataset = MyDataset(train_images, train_masks, transform=image_transforms, target_transform=mask_transforms)
test_dataset = MyDataset(test_images, test_masks, transform=image_transforms, target_transform=mask_transforms)

# Create DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")








# Display 4 images with their masks side by side
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    # TODO: Get an image-mask pair from train_dataset
    # Hint: Use train_dataset[i] to get the i-th sample

    # YOUR CODE HERE
    image, mask = train_dataset[i]

    image = image.permute(1, 2, 0)

    # Display image (denormalize first)
    axes[0, i].imshow(image)
    axes[0, i].set_title(f"Image {i+1}")
    axes[0, i].axis("off")

    # Display mask
    axes[1, i].imshow(mask.squeeze(), cmap="gray")
    axes[1, i].set_title(f"Mask {i+1}")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# TO DO

!pip install -q segmentation_models_pytorch

import segmentation_models_pytorch as smp


# Define U-Net Model
device = "cpu"
model = smp.Unet(
    encoder_name="efficientnet-b1",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=8, # 8 classes for segmenattion
).to(device)

In [ ]:
# TO DO

from tqdm import tqdm    # Shows progress bar

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # Track accuracy
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)  # Get predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy


In [ ]:
# TO DO

import torch
from torch import nn
import torch.optim as optim

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 10  # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")


In [ ]:
# TO DO

import random
import matplotlib.pyplot as plt
import numpy as np

# Function to denormalize images
def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])  # ImageNet mean
    std = np.array([0.229, 0.224, 0.225])  # ImageNet std
    img = img.numpy().transpose(1, 2, 0)  # Convert to HWC
    img = img * std + mean  # Reverse normalization
    img = np.clip(img, 0, 1)  # Clip values to [0,1]
    return img

# Set model to evaluation mode
model.eval()

# Get some test samples
test_samples = random.sample(range(len(test_dataset)), 5)

for idx in test_samples:
    img, mask = test_dataset[idx]

    with torch.no_grad():
        pred_mask = model(img.unsqueeze(0).to(device))  # Forward pass

    pred_mask = (pred_mask >= 0.5).cpu().squeeze().numpy()  # Convert to binary mask

    # Display images
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Original Image (Denormalized)
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    # Ground Truth Mask
    axes[1].imshow(mask.squeeze(), cmap="gray")
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")

    # Predicted Mask
    axes[2].imshow(pred_mask, cmap="gray")
    axes[2].set_title("Predicted Mask")
    axes[2].axis("off")

    plt.show()
